# FASTQ to Fragment File Pipeline (snATAC-seq / scATAC-seq)

This pipeline processes raw ATAC-seq FASTQ files using **Cell Ranger ATAC** (preferred) or **chromap + ArchR/Signac** as a lightweight alternative.

### Tools Used
- **chromap**: Ultra-fast read alignment and barcode correction for ATAC-seq
- Output: fragment files (`.bed.gz`) + barcode lists, ready for ArchR or Signac

### Expected Input
ATAC-seq typically has **3 files per sample**:
- `SAMPLE_R1.fastq.gz` — Read 1 (genomic)
- `SAMPLE_R2.fastq.gz` — Barcode / index read
- `SAMPLE_R3.fastq.gz` — Read 2 (genomic)

Some datasets use only R1 + R2 (barcode in R2). The pipeline handles both.

### Steps
1. **Configure paths** — set input/output directories and reference paths
2. **Test run** — process a single sample to verify the setup
3. **Full run (tmux)** — use the shell script to process all samples

## Cell 1 — Install / Check Dependencies

In [ ]:
import subprocess, shutil, sys

def check_tool(name):
    path = shutil.which(name)
    if path:
        result = subprocess.run([name, '--version'], capture_output=True, text=True)
        version = (result.stdout or result.stderr).strip().splitlines()[0]
        print(f'✅ {name}: {version}')
    else:
        print(f'❌ {name}: NOT FOUND — install before proceeding')

check_tool('chromap')
check_tool('samtools')
check_tool('pigz')   # parallel gzip — optional but fast

# Python packages
for pkg in ['pysam']:
    try:
        __import__(pkg)
        print(f'✅ Python: {pkg} available')
    except ImportError:
        print(f'⚠️  Python: {pkg} not found (optional)')

## Cell 2 — Configure Paths

In [ ]:
import os, glob

# ── USER SETTINGS ─────────────────────────────────────────────────────────────

BASE_DIR   = '/home/nakagawa/datasets'

# Input: directory containing FASTQ files
FASTQ_DIR  = os.path.join(BASE_DIR, 'SRR_cortex/SRR_cortex_ATAC')   # ← change

# Output: where fragment files will be written
OUT_DIR    = os.path.join(BASE_DIR, 'SRR_cortex/SRR_cortex_ATAC/processed_atac')

# Reference genome (chromap index)
# Build with: chromap -i -r genome.fa -o genome.index
GENOME_INDEX = os.path.join(BASE_DIR, 'genome/mm10/chromap_index/genome.index')
GENOME_FA    = os.path.join(BASE_DIR, 'genome/mm10/Mus_musculus.GRCm38.dna.primary_assembly.fa')

# 10x barcode whitelist (choose correct chemistry)
# Common paths (adjust for your conda env):
#   10x ATAC v1: 737K-arc-v1.txt.gz  or  737K-cratac-v1.txt.gz
BARCODE_WHITELIST = os.path.join(BASE_DIR, 'genome/barcodes/737K-cratac-v1.txt.gz')  # ← change

# FASTQ read layout — adjust to match your data:
#   '3-read'  → R1 (gDNA), R2 (barcode 16bp), R3 (gDNA)   ← most 10x ATAC
#   '2-read'  → R1 (gDNA), R2 (barcode+gDNA)              ← some datasets
READ_LAYOUT = '3-read'

# Threads
THREADS = 8

# ── AUTO-DETECT SAMPLES ───────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)

if READ_LAYOUT == '3-read':
    r3_files = sorted(glob.glob(os.path.join(FASTQ_DIR, '*_R3*.fastq.gz')))
    samples = [os.path.basename(f).split('_R3')[0] for f in r3_files]
else:
    r2_files = sorted(glob.glob(os.path.join(FASTQ_DIR, '*_R2*.fastq.gz')))
    samples = [os.path.basename(f).split('_R2')[0] for f in r2_files]

# Remove duplicates while preserving order
seen = set()
samples = [s for s in samples if not (s in seen or seen.add(s))]

print(f'FASTQ_DIR    : {FASTQ_DIR}')
print(f'OUT_DIR      : {OUT_DIR}')
print(f'GENOME_INDEX : {GENOME_INDEX}')
print(f'BARCODE_WL   : {BARCODE_WHITELIST}')
print(f'READ_LAYOUT  : {READ_LAYOUT}')
print(f'THREADS      : {THREADS}')
print(f'\nDetected {len(samples)} sample(s):')
for s in samples:
    print(f'  {s}')

## Cell 3 — Helper: Run One Sample
Define a Python function that calls chromap for a single sample.  
This makes the test run (Cell 4) and full run (Cell 5) share the same logic.

In [ ]:
import subprocess, os

def run_chromap(sample, fastq_dir, out_dir, genome_index, genome_fa,
                barcode_whitelist, read_layout='3-read', threads=8):
    """
    Run chromap for one ATAC-seq sample.
    Returns True if successful, False otherwise.
    """
    sample_out = os.path.join(out_dir, sample)
    os.makedirs(sample_out, exist_ok=True)

    frag_file = os.path.join(sample_out, 'fragments.bed.gz')

    if read_layout == '3-read':
        r1 = os.path.join(fastq_dir, f'{sample}_R1*.fastq.gz')
        r2 = os.path.join(fastq_dir, f'{sample}_R2*.fastq.gz')  # barcode read
        r3 = os.path.join(fastq_dir, f'{sample}_R3*.fastq.gz')
        import glob
        r1 = sorted(glob.glob(r1))
        r2 = sorted(glob.glob(r2))
        r3 = sorted(glob.glob(r3))
        if not r1 or not r2 or not r3:
            print(f'  [SKIP] Missing FASTQ files for {sample}')
            print(f'         R1: {r1}\n         R2 (BC): {r2}\n         R3: {r3}')
            return False
        cmd = [
            'chromap', '--preset', 'atac',
            '-x', genome_index,
            '-r', genome_fa,
            '-1', ','.join(r1),
            '-2', ','.join(r2),   # barcode
            '-3', ','.join(r3),
            '-b', ','.join(r2),   # chromap: -b is barcode FASTQ
            '--barcode-whitelist', barcode_whitelist,
            '-o', frag_file,
            '-t', str(threads),
            '--SAM',              # also output SAM for QC; remove if disk is limited
            '--output-mappings-not-in-whitelist',
        ]
        # Correct: in 3-read layout -1=R1, -2=R3 (both gDNA), -b=R2 (barcode)
        cmd = [
            'chromap', '--preset', 'atac',
            '-x', genome_index,
            '-r', genome_fa,
            '-1', ','.join(r1),
            '-2', ','.join(r3),
            '-b', ','.join(r2),
            '--barcode-whitelist', barcode_whitelist,
            '-o', frag_file,
            '-t', str(threads),
        ]
    else:  # 2-read
        import glob
        r1 = sorted(glob.glob(os.path.join(fastq_dir, f'{sample}_R1*.fastq.gz')))
        r2 = sorted(glob.glob(os.path.join(fastq_dir, f'{sample}_R2*.fastq.gz')))
        if not r1 or not r2:
            print(f'  [SKIP] Missing FASTQ files for {sample}')
            return False
        cmd = [
            'chromap', '--preset', 'atac',
            '-x', genome_index,
            '-r', genome_fa,
            '-1', ','.join(r1),
            '-2', ','.join(r2),
            '--barcode-whitelist', barcode_whitelist,
            '-o', frag_file,
            '-t', str(threads),
        ]

    print(f'  Running: {" ".join(cmd[:6])} ...')
    result = subprocess.run(cmd, capture_output=False, text=True)

    if result.returncode == 0:
        # Index the fragment file for ArchR/Signac
        idx_cmd = ['samtools', 'index', frag_file]
        subprocess.run(idx_cmd)
        print(f'  ✅ Finished: {sample}')
        print(f'     Fragment file: {frag_file}')
        return True
    else:
        print(f'  ❌ FAILED: {sample} (exit code {result.returncode})')
        return False

print('Helper function run_chromap() defined.')

## Cell 4 — 🧪 TEST RUN (single sample)
Run this in Jupyter first to make sure everything works before batch processing.

**Set `TEST_SAMPLE` below to one of your sample names.**

In [ ]:
# ── SET THIS to one sample name from Cell 2 output ──────────────────────────
TEST_SAMPLE = samples[0]   # or e.g. 'SRR12082755'
# ─────────────────────────────────────────────────────────────────────────────

print(f'=== TEST RUN: {TEST_SAMPLE} ===')
success = run_chromap(
    sample           = TEST_SAMPLE,
    fastq_dir        = FASTQ_DIR,
    out_dir          = OUT_DIR,
    genome_index     = GENOME_INDEX,
    genome_fa        = GENOME_FA,
    barcode_whitelist= BARCODE_WHITELIST,
    read_layout      = READ_LAYOUT,
    threads          = THREADS,
)

if success:
    print('\n✅ Test passed — ready for full batch run via tmux script.')
else:
    print('\n❌ Test failed — fix errors above before running all samples.')

## Cell 5 — 📊 QC: Check Output of Test Sample

In [ ]:
import os, subprocess

sample_out = os.path.join(OUT_DIR, TEST_SAMPLE)
frag_file  = os.path.join(sample_out, 'fragments.bed.gz')

if not os.path.exists(frag_file):
    print(f'Fragment file not found: {frag_file}')
else:
    # Count fragments
    result = subprocess.run(
        f'zcat {frag_file} | grep -v "^#" | wc -l',
        shell=True, capture_output=True, text=True
    )
    n_frags = result.stdout.strip()

    # Count unique barcodes
    result2 = subprocess.run(
        f'zcat {frag_file} | grep -v "^#" | cut -f4 | sort -u | wc -l',
        shell=True, capture_output=True, text=True
    )
    n_barcodes = result2.stdout.strip()

    # File size
    size_mb = os.path.getsize(frag_file) / 1e6

    print(f'Fragment file : {frag_file}')
    print(f'File size     : {size_mb:.1f} MB')
    print(f'Total fragments  : {n_frags}')
    print(f'Unique barcodes  : {n_barcodes}')

    # Preview first 5 lines
    print('\nFirst 5 lines of fragment file:')
    preview = subprocess.run(
        f'zcat {frag_file} | grep -v "^#" | head -5',
        shell=True, capture_output=True, text=True
    )
    print(preview.stdout)

## Cell 6 — 💾 Generate tmux Shell Script
Run this cell to write `run_atac.sh` — the script you will run inside tmux for all samples.

**Do NOT run the full batch here in Jupyter.** Use the generated script in tmux instead.

In [ ]:
script_path = os.path.join(os.path.dirname(FASTQ_DIR), 'run_atac.sh')

if READ_LAYOUT == '3-read':
    chromap_read_args = '-1 "$R1" -2 "$R3" -b "$R2"'
    r_detection = '''
    R1=$(ls "$FASTQ_DIR"/${SAMPLE}_R1*.fastq.gz 2>/dev/null | tr '\n' ',' | sed 's/,$//')
    R2=$(ls "$FASTQ_DIR"/${SAMPLE}_R2*.fastq.gz 2>/dev/null | tr '\n' ',' | sed 's/,$//')  # barcode
    R3=$(ls "$FASTQ_DIR"/${SAMPLE}_R3*.fastq.gz 2>/dev/null | tr '\n' ',' | sed 's/,$//')
    if [[ -z "$R1" || -z "$R2" || -z "$R3" ]]; then
        echo "  [SKIP] Missing R1/R2/R3 for $SAMPLE"
        continue
    fi'''
else:
    chromap_read_args = '-1 "$R1" -2 "$R2"'
    r_detection = '''
    R1=$(ls "$FASTQ_DIR"/${SAMPLE}_R1*.fastq.gz 2>/dev/null | tr '\n' ',' | sed 's/,$//')
    R2=$(ls "$FASTQ_DIR"/${SAMPLE}_R2*.fastq.gz 2>/dev/null | tr '\n' ',' | sed 's/,$//')
    if [[ -z "$R1" || -z "$R2" ]]; then
        echo "  [SKIP] Missing R1/R2 for $SAMPLE"
        continue
    fi'''

script_content = f'''#!/bin/bash
# ============================================================
#  ATAC-seq pipeline — chromap batch runner
#  Run inside tmux: bash run_atac.sh
#  Detach with Ctrl+B, D after starting
# ============================================================

FASTQ_DIR="{FASTQ_DIR}"
OUT_DIR="{OUT_DIR}"
GENOME_INDEX="{GENOME_INDEX}"
GENOME_FA="{GENOME_FA}"
BARCODE_WHITELIST="{BARCODE_WHITELIST}"
THREADS={THREADS}
READ_LAYOUT="{READ_LAYOUT}"

mkdir -p "$OUT_DIR"

LOG="$OUT_DIR/pipeline_$(date +%Y%m%d_%H%M%S).log"
exec > >(tee -a "$LOG") 2>&1
echo "Pipeline started: $(date)"
echo "Logging to: $LOG"

# Detect samples from R3 files (3-read) or R2 files (2-read)
if [[ "$READ_LAYOUT" == "3-read" ]]; then
    SAMPLES=($(ls "$FASTQ_DIR"/*_R3*.fastq.gz 2>/dev/null | xargs -I{{}} basename {{}} | sed 's/_R3.*//' | sort -u))
else
    SAMPLES=($(ls "$FASTQ_DIR"/*_R2*.fastq.gz 2>/dev/null | xargs -I{{}} basename {{}} | sed 's/_R2.*//' | sort -u))
fi

echo "Found ${{#SAMPLES[@]}} sample(s): ${{SAMPLES[*]}}"
echo "------------------------------------------------------------"

FAILED=()

for SAMPLE in "${{SAMPLES[@]}}"; do
    echo ""
    echo "============================================================"
    echo "Processing: $SAMPLE   ($(date))"
    echo "============================================================"

    SAMPLE_OUT="$OUT_DIR/$SAMPLE"
    mkdir -p "$SAMPLE_OUT"
    FRAG="$SAMPLE_OUT/fragments.bed.gz"
{r_detection}

    chromap --preset atac \\
        -x "$GENOME_INDEX" \\
        -r "$GENOME_FA" \\
        {chromap_read_args} \\
        --barcode-whitelist "$BARCODE_WHITELIST" \\
        -o "$FRAG" \\
        -t "$THREADS"

    STATUS=$?

    if [[ $STATUS -eq 0 ]]; then
        echo "  ✅ Finished: $SAMPLE"
        samtools index "$FRAG"
    else
        echo "  ❌ FAILED: $SAMPLE (exit $STATUS)"
        FAILED+=("$SAMPLE")
    fi
done

echo ""
echo "============================================================"
echo "Pipeline finished: $(date)"
if [[ ${{#FAILED[@]}} -eq 0 ]]; then
    echo "All samples completed successfully."
else
    echo "Failed samples (${{#FAILED[@]}}):"
    for s in "${{FAILED[@]}}"; do echo "  - $s"; done
fi
echo "Log saved to: $LOG"
'''

with open(script_path, 'w') as f:
    f.write(script_content)

os.chmod(script_path, 0o755)
print(f'✅ Shell script written to: {script_path}')
print()
print('To run all samples, in your tmux terminal:')
print(f'  conda activate scrna   # or whichever env has chromap')
print(f'  bash {script_path}')
print(f'  # then Ctrl+B, D to detach')

---
## Notes

### Building the chromap genome index (first time only)
```bash
conda activate scrna
chromap -i -r /path/to/genome.fa -o /path/to/genome.index
```

### Barcode whitelists
| Chemistry | File |
|---|---|
| 10x ATAC v1 | `737K-cratac-v1.txt.gz` |
| 10x Multiome ATAC | `737K-arc-v1.txt.gz` |

These are bundled with Cell Ranger ATAC. Alternatively download from the 10x Genomics website.

### Loading fragment files in ArchR (R)
```r
library(ArchR)
frags <- list.files('processed_atac', pattern='fragments.bed.gz$', recursive=TRUE, full.names=TRUE)
names(frags) <- basename(dirname(frags))
proj <- createArrowFiles(inputFiles=frags, sampleNames=names(frags))
```

### Loading in Signac (R)
```r
library(Signac)
frags <- CreateFragmentObject(path='processed_atac/SAMPLE/fragments.bed.gz')
```